In [1]:
from pathlib import Path 
import sys 

import numpy as np 
import pandas as pd 

PROJECT_ROOT = Path.cwd() 

if PROJECT_ROOT.name == "notebooks": 
    PROJECT_ROOT = PROJECT_ROOT.parent 

sys.path.insert(0, str(PROJECT_ROOT))

from src.embedder import Embedder 
from src.vector_store import VectorStore

/Users/aob/AOB/CS/Projects/clip-semantic-search/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1108 21:03:50.993000 36937 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
store_path = PROJECT_ROOT / "qdrant_storage" 

vector_store = VectorStore(
    storage_path=str(store_path), 
    collection_name="abo_products", 
    vector_size=512, 
)
print("Stored points:", vector_store.count())

Stored points: 1000


In [3]:
embedder = Embedder() 

print("Device:", embedder.device) 

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 85005.24it/s]


Device: mps


In [4]:
queries = [
    "a black handbag",
    "a blue phone case",
    "modern furniture",
    "home decor",
    "comfortable running shoes",
    "a red handbag",
    "a transparent phone case",
    "a dining chair",
    "bedroom decoration",
    "black sandals",
    "a small crossbody bag",
    "a sofa for a living room",
]

In [6]:
evaluation_rows = [] 

for query in queries: 
    query_embedding = embedder.embed_text(query) 

    results = vector_store.search(
        query_embedding=query_embedding, 
        limit=5, 
    )

    for rank, result in enumerate(results, start=1): 
        evaluation_rows.append(
            {
                "query": query, 
                "rank": rank, 
                "item_id": result["item_id"], 
                "product_type": result["product_type"], 
                "title": result["title"], 
                "score": result["score"], 
                "image_path": result["image_path"], 
            }
        )

evaluation_results = pd.DataFrame(evaluation_rows)

print("Total retrieved rows:", len(evaluation_results))
evaluation_results.head() 

Total retrieved rows: 60


,query,rank,item_id,product_type,title,score,image_path
0,a black handbag,1,B081HMLLBR,HANDBAG,Nia & Nicole Women's Handbag (Navy Blue),0.306439,data/ABO/abo-images-small/images/small/0f/0fa0...
1,a black handbag,2,B000R5H79K,CELLULAR_PHONE_CASE,Rivet The Wrap Fashion Case for Mobile Phones,0.304861,data/ABO/abo-images-small/images/small/5c/5c4c...
2,a black handbag,3,B07WRJFTRK,HANDBAG,Flavia Women's Handbag (Black),0.301635,data/ABO/abo-images-small/images/small/c4/c427...
3,a black handbag,4,B06X17B2N4,HANDBAG,The Lovely bolsa Co. - Bolera de nailon con as...,0.296398,data/ABO/abo-images-small/images/small/ad/add5...
4,a black handbag,5,B08F9ZWWXP,HANDBAG,WindTook Small Crossbody Bag for Women Shoulde...,0.293936,data/ABO/abo-images-small/images/small/f9/f9bd...


In [7]:
for query in queries: 
    print(f"\n{'=' * 80}")
    print(f"QUERY: {query}")
    print("=" * 80) 

    query_results = evaluation_results[
        evaluation_results["query"] == query
    ]

    print(
        query_results[
            ["rank", "product_type", "score", "title"] 
        ].to_string(index=False) 
    )


QUERY: a black handbag
 rank        product_type    score                                                                                                                                                                         title
    1             HANDBAG 0.306439                                                                                                                                      Nia & Nicole Women's Handbag (Navy Blue)
    2 CELLULAR_PHONE_CASE 0.304861                                                                                                                                 Rivet The Wrap Fashion Case for Mobile Phones
    3             HANDBAG 0.301635                                                                                                                                                Flavia Women's Handbag (Black)
    4             HANDBAG 0.296398 The Lovely bolsa Co. - Bolera de nailon con asa superior a rayas en contraste con pompones de piel sintét

In [8]:
evaluation_dir = PROJECT_ROOT / "data" / "evaluation" 
evaluation_dir.mkdir(parents=True, exist_ok=True) 

results_path = evaluation_dir / "retrieval_results.csv" 

evaluation_results.to_csv(results_path, index=False) 

print(f"Saved results to: {results_path}") 

Saved results to: /Users/aob/AOB/CS/Projects/clip-semantic-search/data/evaluation/retrieval_results.csv


In [10]:
vector_store.close() 

In [11]:
relevance_labels = {
    "a black handbag": [0, 0, 1, 1, 1],
    "a blue phone case": [1, 1, 1, 1, 0],
    "modern furniture": [1, 1, 1, 1, 1],
    "home decor": [1, 1, 1, 1, 1],
    "comfortable running shoes": [0, 0, 0, 0, 0],
    "a red handbag": [1, 1, 1, 1, 1],
    "a transparent phone case": [1, 1, 1, 1, 1],
    "a dining chair": [1, 1, 1, 1, 1],
    "bedroom decoration": [1, 1, 1, 1, 1],
    "black sandals": [1, 1, 1, 1, 1],
    "a small crossbody bag": [0, 1, 1, 1, 1],
    "a sofa for a living room": [1, 0, 1, 1, 0],
}

In [12]:
if set(relevance_labels) != set(queries): 
    raise ValueError("Every query must have relevanc labels.")

if any(len(labels) != 5 for labels in relevance_labels.values()): 
    raise ValueError("Every query needs exactly five labels.") 

if any(
    label not in {0,1} 
    for labels in relevance_labels.values() 
    for label in labels 
): 
    raise ValueError("Labels must be either 0 or 1.") 

print("Relevance labels are valid.")

Relevance labels are valid.


In [14]:
evaluation_results["relevant"] = [
    relevance_labels[query][rank - 1] 
    for query, rank in zip(
        evaluation_results["query"], 
        evaluation_results["rank"], 
    )
]


In [15]:
evaluation_results

,query,rank,item_id,product_type,title,score,image_path,relevant
0,a black handbag,1,B081HMLLBR,HANDBAG,Nia & Nicole Women's Handbag (Navy Blue),0.306439,data/ABO/abo-images-small/images/small/0f/0fa0...,0
1,a black handbag,2,B000R5H79K,CELLULAR_PHONE_CASE,Rivet The Wrap Fashion Case for Mobile Phones,0.304861,data/ABO/abo-images-small/images/small/5c/5c4c...,0
2,a black handbag,3,B07WRJFTRK,HANDBAG,Flavia Women's Handbag (Black),0.301635,data/ABO/abo-images-small/images/small/c4/c427...,1
3,a black handbag,4,B06X17B2N4,HANDBAG,The Lovely bolsa Co. - Bolera de nailon con as...,0.296398,data/ABO/abo-images-small/images/small/ad/add5...,1
4,a black handbag,5,B08F9ZWWXP,HANDBAG,WindTook Small Crossbody Bag for Women Shoulde...,0.293936,data/ABO/abo-images-small/images/small/f9/f9bd...,1
5,a blue phone case,1,B082DNQR4W,CELLULAR_PHONE_CASE,Amazon Brand - Solimo Soft & Flexible Back Pho...,0.333410,data/ABO/abo-images-small/images/small/bb/bb64...,1
6,a blue phone case,2,B08562DD6Z,CELLULAR_PHONE_CASE,Amazon Brand - Solimo Soft & Flexible Back Pho...,0.309280,data/ABO/abo-images-small/images/small/b6/b663...,1
7,a blue phone case,3,B079J6N6PK,CELLULAR_PHONE_CASE,Amazon Brand - Solimo Mobile Cover (Soft & Fle...,0.304798,data/ABO/abo-images-small/images/small/90/909c...,1
8,a blue phone case,4,B07VXRRB3J,CELLULAR_PHONE_CASE,Amazon Brand - Solimo Redmi 7A Mobile Cover (H...,0.303270,data/ABO/abo-images-small/images/small/b4/b491...,1
9,a blue phone case,5,B004I8WD7S,CELLULAR_PHONE_CASE,AmazonBasics Protective TPU Case with Screen P...,0.302422,data/ABO/abo-images-small/images/small/45/45b2...,0


In [16]:
# Calculate precision@5 

precision_by_query = (
    evaluation_results 
    .groupby("query")["relevant"] 
    .mean() 
    .reset_index(name="precision_at_5")
)

precision_by_query

,query,precision_at_5
0,a black handbag,0.6
1,a blue phone case,0.8
2,a dining chair,1.0
3,a red handbag,1.0
4,a small crossbody bag,0.8
5,a sofa for a living room,0.6
6,a transparent phone case,1.0
7,bedroom decoration,1.0
8,black sandals,1.0
9,comfortable running shoes,0.0


In [17]:
overall_precision_at_5 = evaluation_results["relevant"].mean() 

print(
    f"Overall precision@5: "
    f"{overall_precision_at_5:.3f}"
)

Overall precision@5: 0.817


In [18]:
labeled_path = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "retrieval_results_labeled.csv"
)

evaluation_results.to_csv(labeled_path, index=False)

print(f"Saved labeled results to: {labeled_path}")

Saved labeled results to: /Users/aob/AOB/CS/Projects/clip-semantic-search/data/evaluation/retrieval_results_labeled.csv


In [19]:
summary_path = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "precision_at_5_by_query.csv"
)

precision_by_query.to_csv(summary_path, index=False)